# 17 — Transactions, Constraints, and Exactly-Once Semantics

Atomicity matters more than application-level checks under concurrency. Practice schema-enforced uniqueness, rollback, idempotency records, and failure-safe multi-step updates.

*Advanced · 90–150 min*

## You will learn
- Enforce constraints in a real database rather than only memory
- Use SQLite transactions correctly
- Design a persistent idempotency key table
- Recognize the difference between at-least-once delivery and exactly-once effects

## Examples

### 1. Atomic transactions

A multi-step transfer should either update both accounts or neither. Use a transaction context manager and a CHECK constraint; if any statement fails, the transaction rolls back. Close the connection explicitly.

In [ ]:
import sqlite3

db = sqlite3.connect(":memory:")
try:
    db.execute("CREATE TABLE accounts (id INTEGER PRIMARY KEY, balance INTEGER CHECK(balance >= 0))")
    db.executemany("INSERT INTO accounts VALUES (?, ?)", [(1, 100), (2, 20)])
    db.commit()
    try:
        with db:
            db.execute("UPDATE accounts SET balance = balance - 150 WHERE id = 1")
            db.execute("UPDATE accounts SET balance = balance + 150 WHERE id = 2")
    except sqlite3.IntegrityError:
        print("Transfer rolled back")
    print(db.execute("SELECT id, balance FROM accounts ORDER BY id").fetchall())
finally:
    db.close()

**Review:** What happens if the sender has insufficient funds during a real transfer?

### 2. Unique constraints beat check-then-insert

Checking for duplicates before insertion is not atomic under concurrency. A UNIQUE constraint can enforce one active identifier at storage level, and conflicts can be handled deliberately.

In [ ]:
import sqlite3
db = sqlite3.connect(":memory:")
try:
    db.execute("CREATE TABLE jobs (job_id TEXT PRIMARY KEY, status TEXT NOT NULL)")
    db.execute("INSERT INTO jobs VALUES (?, ?)", ("J1", "queued"))
    try:
        db.execute("INSERT INTO jobs VALUES (?, ?)", ("J1", "queued"))
    except sqlite3.IntegrityError:
        print("Duplicate job rejected")
finally:
    db.close()

**Review:** Why might an in-memory set of seen job IDs fail in a multi-instance service?

### 3. Idempotency in persistent storage

A durable idempotency table can associate a logical operation key with a stored result. The key and side effect should be committed atomically; otherwise retrying after a crash can duplicate the effect.

In [ ]:
import sqlite3
db = sqlite3.connect(":memory:")
try:
    db.execute("CREATE TABLE outcomes (request_key TEXT PRIMARY KEY, result TEXT NOT NULL)")
    def process(key, outcome):
        with db:
            db.execute("INSERT OR IGNORE INTO outcomes VALUES (?, ?)", (key, outcome))
            return db.execute("SELECT result FROM outcomes WHERE request_key=?", (key,)).fetchone()[0]
    print(process("order-1", "confirmed"))
    print(process("order-1", "different"))
finally:
    db.close()

**Review:** What validation is needed if the same key is reused with a different request body?

### 4. Resource cleanup and reliability

The sqlite3 connection context manager commits or rolls back but does not close the connection. Keep ownership explicit, and use connection.close in finally or contextlib.closing. Do not treat network calls inside database transactions casually.

In [ ]:
from contextlib import closing
import sqlite3

with closing(sqlite3.connect(":memory:")) as connection:
    with connection:
        connection.execute("CREATE TABLE events (id INTEGER PRIMARY KEY, name TEXT)")
        connection.execute("INSERT INTO events (name) VALUES (?)", ("ready",))
    count = connection.execute("SELECT COUNT(*) FROM events").fetchone()[0]
print("Rows:", count)

**Review:** Why should a transaction avoid waiting on slow external HTTP requests?

## Try it

Short fill-in exercises on this lesson: replace each `___`, run the cell, and `check` tells you whether you're right. The answer itself stays hidden. More practice for this topic is in the [workbook](../exercises/04_production_systems_exercises.ipynb).

In [ ]:
from lesson import Exercises, ___, run

check = Exercises('advanced/17', {1: '20516f45934a5a04', 2: 'b4f72e1ec638db06', 3: '8b0666f0dc447fcd', 4: '62a05d08a180f75d'})

**Exercise 1.** A transfer that enforces the non-negative balance invariant atomically.

In [ ]:
import sqlite3
conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE acct (name TEXT PRIMARY KEY, balance INTEGER NOT NULL CHECK (balance >= ___))")
conn.execute("INSERT INTO acct VALUES ('a', 100), ('b', 0)"); conn.commit()

def transfer(src, dst, amount):
    try:
        with conn:
            conn.execute("UPDATE acct SET balance = balance + ? WHERE name = ?", (amount, dst))
            conn.execute("UPDATE acct SET balance = balance - ? WHERE name = ?", (amount, src))
        return "ok"
    except sqlite3.IntegrityError:
        return "rejected"

check(1, [transfer("a", "b", 60), transfer("a", "b", 60), dict(conn.execute("SELECT * FROM acct"))])

**Exercise 2.** An idempotency table rejects a different payload for the same key.

In [ ]:
import sqlite3
conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE idem (key TEXT PRIMARY KEY, payload TEXT NOT NULL)")

def record(key, payload):
    row = conn.execute("SELECT payload FROM idem WHERE key = ?", (key,)).fetchone()
    if row:
        return "duplicate" if row[0] == ___ else "conflict"
    with conn:
        conn.execute("INSERT INTO idem VALUES (?, ?)", (key, payload))
    return "stored"

check(2, [record("k1", "pay 10"), record("k1", "pay 10"), record("k1", "pay 99")])

**Exercise 3.** If the second statement fails, the first one must be rolled back too.

In [ ]:
import sqlite3
conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE t (id INTEGER PRIMARY KEY)")
try:
    with conn:
        conn.execute("INSERT INTO t VALUES (1)")
        conn.execute("INSERT INTO t VALUES (1)")     # duplicate primary key
except sqlite3.___:
    pass
check(3, conn.execute("SELECT COUNT(*) FROM t").fetchone()[0])

**Exercise 4.** Exactly-once *effects* = at-least-once delivery + deduplication. Process duplicated deliveries.

In [ ]:
deliveries = ["m1", "m2", "m1", "m3", "m2"]
processed, effects = set(), []
for message_id in deliveries:
    if message_id in ___:
        continue
    processed.add(message_id)
    effects.append(message_id)
check(4, effects)

## Common mistakes

- check-then-insert alone can race under concurrent callers.
- A transaction context does not automatically close a SQLite connection.
- Exactly-once effects require carefully coordinated durable state and duplicate handling.

## Quick quiz

Replace each `'?'` in the cell below with your choice and run it. The answers are hidden, but each one is checked.

**1.** Why enforce uniqueness in the database, not just Python?

   - **a)** Python can't compare strings
   - **b)** Concurrent writers can bypass Python checks
   - **c)** It's prettier

**2.** At-least-once delivery means a message may be…

   - **a)** Encrypted
   - **b)** Processed more than once
   - **c)** Lost

**3.** To make processing effectively exactly-once, combine at-least-once with…

   - **a)** Idempotent writes (e.g. unique keys)
   - **b)** Bigger servers
   - **c)** Retries

In [ ]:
from quiz import Quiz

quiz = Quiz('advanced/17', {1: '2d20b504f349', 2: '22cdefbd6b25', 3: 'ba459d8366c4'})
quiz.answer(1, '?')
quiz.answer(2, '?')
quiz.answer(3, '?')

**Next:** [18 — Production Quality: Packaging, CI, Configuration, and Deployment](18_production_quality_ci_deployment.ipynb)